In [24]:
# import all necessary libraries
import numpy as np
import pandas as pd
import h5py
import vaex
import pynbody
from pynbody.array import SimArray
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.patches import PathPatch
from matplotlib.path import Path
from astropy import units as u
from astropy.io import ascii, fits
from astropy.table import Table, vstack
from astropy.coordinates import SkyCoord,CartesianRepresentation,match_coordinates_sky
import functions as fn 
import matched_filter_ani as mf
import os
from pathlib import Path as path

np.random.seed(0)

In [10]:
num = 5 # second half (number) of the galaxy name.

# import the snapshot
s = pynbody.load('/home/christenc/Storage/Cosmo/rogue.cosmo25cmb/rogue.cosmo25cmb.4096g5HbwK1BH/rogue.cosmo25cmb.4096g5HbwK1BH.004096/rogue.cosmo25cmb.4096g5HbwK1BH.004096')

# access the halos and their unique IDs
h = s.halos(halo_numbers='v1')  # load the halos using the original AHF numbering system
h.load_all()
unique_halo_ids = list(h.keys()) # mixed in terms of size

# identify the main halo (halo with most stars)
main_halo = h[num] # for Marvelous Massive, h[1]. For Marvel, h[num].

# identify the center of the galaxy
pynbody.analysis.halo.center
cen = main_halo.mean_by_mass('pos')

# import all the star particles (sp) within a radius of 200kpc from the center of the galaxy
sp = s[pynbody.filt.Sphere(SimArray([200], "kpc"), cen)].load_copy()

# initiate physical units
s.physical_units()

# access particles IDs
partids_snap = sp.s['iord']

In [11]:
# loads h5 data. Snapshot has locations, h5 has progenitor info.
with h5py.File('/home/christenc/Storage/Cosmo/rogue.cosmo25cmb/rogue.cosmo25cmb.4096g5HbwK1BH/rogue.cosmo25cmb.4096g5HbwK1BH_allhalostardata_upd.h5','r') as f:
    hostids = f['host_IDs'].asstr()[:] 
    partids_h5 = f['particle_IDs'][:]

In [22]:
# define the name of the galaxy, its depth, distance, resolution, parameters and filepaths
box = 'rogue'
D = 2000 
mlim_str = '26p5' 
name = f'{box}_4096_{num}_data_{D}_{mlim_str}'
depth=26
distance=2000

dwarfcatpath = f"/home/giantsid/MAP/raw_data/{box}_{num}/survey.{name}.0.h5" # Marvel convention
#dwarfcatpath = f"/home/otteleno/MAP/raw_data/{box}_{num}/survey.MM_{box}{num}_data_2000_26p5.0.h5" # Marvelous Massive convention
vdwarfcat = vaex.open(dwarfcatpath)
dwarfcat = pd.DataFrame(vdwarfcat,columns=vdwarfcat.column_names)

size_kpc = 40 
pdist = (size_kpc/D) * (180/np.pi) 
year = 10
mlim = '25'
c1='px'
c2='py'
edgelength = 10 
plotdir = f'{box}_4096_{num}' 

In [13]:
dwarfcat

,age,dec,dmod,feh,glat,glon,grav,lsst_g,lsst_g_Err,lsst_g_Intrinsic,...,py,pz,ra,rad,smass,teff,vr,vx,vy,vz
0,10.116360,-27.115203,26.508263,-1.864954,-89.791569,36.468260,1.708200,26.129591,-0.0,-0.378671,...,4.330650,-2002.855621,12.625756,2002.868874,0.789576,4888.910156,112.689970,-78.721655,-284.048948,-113.535196
1,10.116075,-27.068847,26.512727,-1.918716,-89.829245,-167.385995,2.029733,26.736753,-0.0,0.224026,...,-1.306205,-2006.981939,13.039307,2006.990852,0.789244,5021.247070,45.183771,-91.040930,-417.871271,-44.647233
2,10.115990,-27.044058,26.511161,-1.975372,-89.861106,175.646536,2.064576,26.801401,-0.0,0.290241,...,0.369051,-2005.538203,12.983552,2005.544096,0.789206,5036.276367,29.748764,-142.495139,-350.306650,-29.468879
3,10.115932,-26.904219,26.506917,-1.783898,-89.638948,71.212401,2.015545,26.780186,-0.0,0.273269,...,11.941232,-2001.588015,12.541657,2001.627757,0.795284,4955.287109,126.110261,-132.393378,-308.899095,-128.224313
4,10.115875,-26.722693,26.517836,-2.125204,-89.460042,164.337458,0.937987,24.925571,0.0,-1.592265,...,5.118176,-2011.629239,13.259299,2011.718571,0.789853,4534.572266,7.149410,-115.307402,-354.725214,-7.005909
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
343412,5.899282,-27.116665,26.506168,-0.478926,-89.983163,169.446763,4.020476,21.726461,0.0,-4.779706,...,0.107690,-2000.937316,12.873202,2000.937403,26.070622,37819.714844,82.630723,-90.388338,-370.045860,-82.624530
343413,5.899282,-27.116373,26.506155,-0.478926,-89.983129,168.178901,4.384905,25.384672,0.0,-1.121482,...,0.120701,-2000.925669,12.872939,2000.925756,6.206314,21371.732422,83.040292,-89.984966,-368.926655,-83.036615
343414,5.899282,-27.116466,26.506166,-0.478926,-89.983019,168.978042,4.415252,25.822878,-0.0,-0.683288,...,0.113378,-2000.936152,12.873212,2000.936240,5.202567,19545.900391,82.662312,-90.495573,-369.862445,-82.656947
343415,5.899282,-27.116438,26.506148,-0.478926,-89.982953,169.065264,4.115353,22.255793,-0.0,-4.250355,...,0.112931,-2000.919529,12.873286,2000.919618,21.128011,36096.062500,83.546790,-90.256948,-369.944985,-83.541307


In [14]:
# adjust ananke STAR data so that center of mass is the origin.
ananke_data = np.column_stack([dwarfcat['px'],dwarfcat['py'], dwarfcat['pz']])
ananke_masses = dwarfcat['smass']
ananke_total_mass = np.sum(ananke_masses)

cm_current_ananke = [0,0,0]
for i in range(len(ananke_data)):
    cm_current_ananke = cm_current_ananke + ananke_data[i] * ananke_masses[i]
cm_ananke = [x/ananke_total_mass for x in cm_current_ananke]

ananke_data_centered = ananke_data - cm_ananke

In [15]:
def fibonacci_angler(n):
    """ Generates n evenly spaced angles in spherical polar coordinates.

    Args:
        n (int):
            Number of desired evenly spaced angles.

    Returns:
        declination_array ((array of) float(s)):
            The declination(s) of the generated angle(s).
        azimuthal_array ((array of) float(s)):
            The azimuth(s) of the generated angle(s).
    """

    golden_ratio = (1 + np.sqrt(5))/2

    i_array = np.linspace(0,n-1,n)
    z_array = 1 - i_array/((n-1)) #only goes halfway down sphere. intentional
    radius_array = np.sqrt(1-z_array**2)

    declination_array = np.pi/2-np.arcsin(z_array)
    azimuthal_array = 2*np.pi * i_array/golden_ratio

    return declination_array, azimuthal_array

In [16]:
declination_array, azimuthal_array = fibonacci_angler(24) # store the angles

In [18]:
# round angles
d_r = [0] *24
a_r = [0]*24
for i in range(len(declination_array)):
    d_r[i] = round(declination_array[i]*180/np.pi,1)
    a_r[i] = round(np.mod(azimuthal_array[i]*180/np.pi,360),1)

print(d_r)
print(a_r)

[0.0, 17.0, 24.1, 29.6, 34.3, 38.5, 42.3, 45.9, 49.3, 52.5, 55.6, 58.6, 61.4, 64.2, 67.0, 69.6, 72.3, 74.9, 77.4, 80.0, 82.5, 85.0, 87.5, 90.0]
[0.0, 222.5, 85.0, 307.5, 170.0, 32.5, 255.0, 117.4, 339.9, 202.4, 64.9, 287.4, 149.9, 12.4, 234.9, 97.4, 319.9, 182.4, 44.9, 267.4, 129.8, 352.3, 214.8, 77.3]


In [19]:
def new_axes(host_id):
    """ Defines a new coordinate system based on the particles of the given host where
        Z' is the angular momentum vector of the host,
        Y' is the cross product of Z' with the center of mass vector of the host and
        X' is the cross product of Z' with Y'.

    Args:
        host_ids (string):
            The ID of a host.

    Returns:
        transform_matrix (2D 3x3 array of floats):
            The matrix to transform the galaxy to.
            
    """
    # assign particle data to more convenient variables
    m = sp.s['mass']
    x = sp.s['pos'][:, 0]
    y = sp.s['pos'][:, 1]
    z = sp.s['pos'][:, 2]
    vx = sp.s['vel'][:, 0]
    vy = sp.s['vel'][:, 1]
    vz = sp.s['vel'][:, 2]

    particle_array_all = np.column_stack((m,x,y,z,vx,vy,vz))
    pos_array_all = particle_array_all[:, 1:4]
    vel_array_all = particle_array_all[:, 4:7]

    # center particle data and change velocity to that seen from origin
    total_mass_all=np.sum(m)
    
    cm_current_all = [0,0,0]
    for i in range(len(particle_array_all)):
        cm_current_all = cm_current_all + particle_array_all[i][0] * pos_array_all[i]
    cm_all = cm_current_all/total_mass_all
    
    avg_vel_current_all = [0,0,0]
    for i in range(len(particle_array_all)):
        avg_vel_current_all = avg_vel_current_all + particle_array_all[i][0] * vel_array_all[i]
    avg_vel_all = avg_vel_current_all/total_mass_all
    
    pos_array_centered_all = pos_array_all - cm_all
    vel_array_centered_all = vel_array_all - avg_vel_all
    
    # mask to select only particles from the progenitor of interest
    _, idloc_snap, idloc_h5 = np.intersect1d(partids_snap, partids_h5, return_indices = True)                                                                                 

    progenitor = hostids[idloc_h5] 
    mask = np.isin(progenitor, host_id) 
    
    m_halo = m[idloc_snap][mask] 
    x_halo = x[idloc_snap][mask]
    y_halo = y[idloc_snap][mask]
    z_halo = z[idloc_snap][mask]
    vx_halo = vx[idloc_snap][mask]
    vy_halo = vy[idloc_snap][mask]
    vz_halo = vz[idloc_snap][mask]

    # adjust the particles from the progenitor of interest to also be centered and velocity-adjusted
    particle_array_halo = np.column_stack((m_halo,x_halo,y_halo,z_halo,vx_halo,vy_halo,vz_halo)) 
    pos_array_halo = particle_array_halo[:, 1:4]
    vel_array_halo = particle_array_halo[:, 4:7]
    pos_array_centered_halo = pos_array_halo - cm_all 
    vel_array_centered_halo = vel_array_halo - avg_vel_all 
    
    total_mass_halo=np.sum(m_halo)

    cm_current_halo = [0,0,0]
    for i in range(len(particle_array_halo)):
        cm_current_halo = cm_current_halo + particle_array_halo[i][0] * pos_array_centered_halo[i]
    cm_halo = [x/total_mass_halo for x in cm_current_halo]
    
    ang_mom_halo = [0,0,0]
    for i in range(len(particle_array_halo)):
        ang_mom_halo = ang_mom_halo + particle_array_halo[i][0] * np.cross(pos_array_centered_halo[i], vel_array_centered_halo[i])

    # create new basis and write it as a matrix (3x3, rows are basis vectors)
    z_prime = ang_mom_halo/np.linalg.norm(ang_mom_halo)
    y_prime_unnormed = np.cross(ang_mom_halo, cm_halo)
    y_prime =  y_prime_unnormed/np.linalg.norm(y_prime_unnormed)
    x_prime = np.cross(z_prime, y_prime)
    transform_matrix = np.row_stack((x_prime, y_prime, z_prime)) 
    
    return transform_matrix

In [25]:
# save the transform matrix as a global variable
transform_matrix = new_axes("2784_365")

In [26]:
print(transform_matrix)

[[ 0.27640503 -0.49252341  0.82523993]
 [-0.69105502 -0.6985988  -0.18547961]
 [-0.66786468  0.5190187   0.53345699]]


In [27]:
def angle_view(coordinate_transform, declination, azimuthal, save = False, plot=True):
    """Calculates 2d image coordinates based on angle of view. Optionally plots graph. 

    Args:
        coordinate_transform (2D 3x3 array of floats):
            The desired transformation matrix.
        host_ids (array of strings):
            hostid(s) of halo to calibrate axes around, as well as color differently.
        declination (float):
            Angle of observation measured down from z axis.
        azimuthal (float):
            Angle of observation measured counterclockwise from x axis in xy plane.
    
    Optional Args:
        save (boolean):
            Whether to save plot to file system or not.
        plot (boolean):
            Whether to display plot or not.

    Returns:
        declinations_readable (array of floats):
            The declination values with precision of one decimal.
        azimuthal_readable (array of floats):
            The azimuth values with precision of one decimal.
        project_pos_array_all (2D array of floats):
            The new positions of each particle after the galaxy is transformed.
    """
    
    # transform STAR data into new axes.
    new_pos_array_all = np.matmul(coordinate_transform, ananke_data_centered.T)

    # spherical coordinates. Turns into radians.
    theta = declination
    phi = azimuthal 
    declination_readable = round(declination*180/np.pi,1)
    azimuthal_readable = round(np.mod(azimuthal*180/np.pi, 360),1)

    # use rotation matrix to find 2d projected image from any given angle of observation
    project_matrix = np.array([[-np.sin(phi),                np.cos(phi),             0            ],
                             [-np.cos(theta)*np.cos(phi), -np.cos(theta)*np.sin(phi), np.sin(theta)]])

    project_pos_array_all = np.matmul(project_matrix, new_pos_array_all) #applies 2x3 transformation to 3xn data. Result is 2xn data 

    if plot == True:
        fig,ax  = plt.subplots()
        ax.scatter (project_pos_array_all[0], project_pos_array_all[1], s=1)
        ax.set_title(f"Declination = {declination_readable} degrees and Azimuthal = {azimuthal_readable} degrees")
        ax.set_title
        plt.show()
       
    return declination_readable, azimuthal_readable, project_pos_array_all

In [29]:
for i in range(len(declination_array)):

    # save new files, one for each angle of observation. Files contain transformed data for dwarf galaxy, but not background.
    d, a, altered_positions=  angle_view(transform_matrix, declination_array[i], azimuthal_array[i], plot=False)
    dwarfcat['px'] = altered_positions.T[:,0]
    dwarfcat['py'] = altered_positions.T[:,1]

    file_path = path(f"/home/giantsid/MAP/matched_filter/mf_data/rotated_catalogs/{depth}_{distance}/{box}_{num}/{box}_{num}_d={d}_a={a}.h5")
    file_path.parent.mkdir(parents=True, exist_ok=True)

    vframe_new = vaex.from_pandas(dwarfcat)
    vframe_new.export_hdf5(file_path, progress=True)

export(hdf5) [########################################] 100.00% elapsed time  :     0.59s =  0.0m =  0.0h
export(hdf5) [########################################] 100.00% elapsed time  :     0.52s =  0.0m =  0.0h
export(hdf5) [########################################] 100.00% elapsed time  :     0.45s =  0.0m =  0.0h
export(hdf5) [########################################] 100.00% elapsed time  :     0.49s =  0.0m =  0.0h
export(hdf5) [########################################] 100.00% elapsed time  :     0.44s =  0.0m =  0.0h
export(hdf5) [########################################] 100.00% elapsed time  :     0.45s =  0.0m =  0.0h
export(hdf5) [########################################] 100.00% elapsed time  :     0.44s =  0.0m =  0.0h
export(hdf5) [########################################] 100.00% elapsed time  :     0.46s =  0.0m =  0.0h
export(hdf5) [########################################] 100.00% elapsed time  :     0.47s =  0.0m =  0.0h
export(hdf5) [################################